In [1]:
import os
os.chdir("..")

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from recsys_lakehouse.spark import spark_builder
from recsys_lakehouse.lakehouse.operator import TableOperator
from recsys_lakehouse.lakehouse import layers

from pathlib import Path
from pyspark.sql import functions as F, SparkSession, DataFrame
from pyspark.sql.functions import from_unixtime, col, sum, rand, when
from pyspark.sql.types import FloatType, IntegerType, TimestampType, StructType, StructField, StringType

from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS

In [6]:
spark = SparkSession.builder.appName("").master("local[*]").getOrCreate()  # type: ignore

In [7]:
ALS.load(".datalake/model_registry/als_model")

Py4JJavaError: An error occurred while calling o29.load.
: java.lang.NoSuchMethodException: org.apache.spark.ml.recommendation.ALSModel.<init>(java.lang.String)
	at java.base/java.lang.Class.getConstructor0(Class.java:3585)
	at java.base/java.lang.Class.getConstructor(Class.java:2271)
	at org.apache.spark.ml.util.DefaultParamsReader.load(ReadWrite.scala:468)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:77)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.base/java.lang.reflect.Method.invoke(Method.java:569)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
	at py4j.Gateway.invoke(Gateway.java:282)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.CallCommand.ex

In [7]:
dataset_name = "amazon_books_sample10000"

operator = TableOperator(spark)
gold = layers.Gold(dataset_name=dataset_name)

In [9]:
reviews_df = operator.read_table(gold, table=gold.tables['reviews'])

In [10]:
reviews_df.show(5)

+-------------------+--------------------+-----------+------+
|          timestamp|             user_id|parent_asin|rating|
+-------------------+--------------------+-----------+------+
|2013-05-31 08:18:08|AFWVN52MRBWOTIK7U...| 1400074312|   4.0|
|2013-05-30 07:58:32|AFWVN52MRBWOTIK7U...| 0399158243|   3.0|
|2013-05-29 18:15:32|AFWVN52MRBWOTIK7U...| 0062105620|   4.0|
|2013-05-29 08:42:05|AFWVN52MRBWOTIK7U...| 0761453830|   4.0|
|2013-05-29 01:11:54|AFWVN52MRBWOTIK7U...| 0544026888|   4.0|
+-------------------+--------------------+-----------+------+
only showing top 5 rows



In [ ]:
# 1st method map
# from pyspark.ml.feature import StringIndexer

# indexer = StringIndexer(inputCol="user_id", outputCol="user_id_index")
# indexed_df = indexer.fit(reviews_df).transform(reviews_df)
# indexed_df.show()

In [71]:
def build_mapping_df(df: DataFrame, col: str, mapping_suffix: str = "_index") -> DataFrame:
    if col not in df.columns:
        raise ValueError(f"Column {col} not in DataFrame")
    
    mapping_rdd = df.select(col).rdd.map(lambda row: row[col]).distinct().zipWithIndex()
    mapping_df = mapping_rdd.toDF([col, col + mapping_suffix])
    return mapping_df

In [72]:
user_mapping = build_mapping_df(reviews_df, col="user_id")
item_mapping = build_mapping_df(reviews_df, col="parent_asin")

In [73]:
def apply_mapping(df: DataFrame, **kwargs) -> DataFrame:
    for col, mapping_df in kwargs.items():
        mapping_col = next(iter(mapping_df.drop(col).columns), None)
        if mapping_col is None:
            raise ValueError(f"Column {col} not in mapping DataFrame")
        
        df = df.join(mapping_df, on=col, how="left")
        df = df.drop(col)
        df = df.withColumnRenamed(mapping_col, col)
    return df

In [76]:
reviews_df_mapped = apply_mapping(reviews_df, user_id=user_mapping, parent_asin=item_mapping)

In [77]:
reviews_df_mapped.show(5)

+-------------------+------+-------+-----------+
|          timestamp|rating|user_id|parent_asin|
+-------------------+------+-------+-----------+
|2015-04-24 02:43:29|   3.0|    241|       1219|
|2014-12-10 07:35:01|   5.0|     43|       4522|
|2015-04-20 03:34:54|   5.0|    241|       3089|
|2021-04-15 02:38:07|   5.0|    328|       8095|
|2021-04-15 02:36:54|   1.0|    328|       7042|
+-------------------+------+-------+-----------+
only showing top 5 rows



In [81]:
(train_df, valid_df, test_df) = reviews_df_mapped.randomSplit([0.7, 0.2, 0.1])

In [93]:
als = ALS(maxIter=10, rank=16, regParam=0.01, userCol="user_id", itemCol="parent_asin", ratingCol="rating",
          coldStartStrategy="drop")
model = als.fit(train_df)

In [98]:
train_df_preds = model.transform(train_df)
valid_df_preds = model.transform(valid_df)

In [99]:
evaluator = RegressionEvaluator(metricName="rmse", predictionCol="prediction", labelCol="rating")
evaluator.evaluate(train_df_preds), evaluator.evaluate(valid_df_preds)

(0.004288283768035206, 3.9399031870953998)

In [89]:
user_recommendations = model.recommendForAllUsers(10)

In [92]:
user_recommendations.first()

Row(user_id=1, recommendations=[Row(parent_asin=4711, rating=4.999358654022217), Row(parent_asin=1391, rating=4.999350070953369), Row(parent_asin=9484, rating=4.997368812561035), Row(parent_asin=9430, rating=4.997368812561035), Row(parent_asin=9277, rating=4.997368812561035), Row(parent_asin=8975, rating=4.997368812561035), Row(parent_asin=8614, rating=4.997368812561035), Row(parent_asin=8536, rating=4.997368812561035), Row(parent_asin=8486, rating=4.997368812561035), Row(parent_asin=6488, rating=4.997368812561035)])